# Phase 6 — Final Dataset and Training Configuration Review

Audits and removes only exact duplicate annotation lines in the current `final_dataset_v1`, runs full validation, and produces final readiness recommendations. It does not create another dataset version or start training.

In [1]:
from pathlib import Path
import json
import math
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.dataset_review import audit_duplicate_annotations, class_balance_summary, remove_duplicate_annotations, write_duplicate_repair_csv
from weapon_threat_detection.engineering import MERGED_CLASS_NAMES, analyze_person_class, calculate_class_weights, collect_dataset_statistics
from weapon_threat_detection.model_engineering import load_yaml
from weapon_threat_detection.validation import build_integrity_report, summarize_records, validate_dataset, write_validation_csv

config = load_config(ROOT / 'configs' / 'project.yaml')
training = load_yaml(ROOT / 'configs' / 'training.yaml')
experiment = load_yaml(ROOT / 'configs' / 'experiment.yaml')['experiment']
dataset_root = ROOT / experiment['final_dataset']
reports_dir = ROOT / experiment['reports_directory']
logger = configure_logger(ROOT / experiment['logs_directory'], 'final_training_readiness')
before_audit = audit_duplicate_annotations(dataset_root, config['dataset']['splits'])
repair = remove_duplicate_annotations(dataset_root, config['dataset']['splits'])
after_audit = audit_duplicate_annotations(dataset_root, config['dataset']['splits'])
if after_audit['duplicate_annotations'] != 0:
    raise RuntimeError(f'Duplicate annotation audit failed: {after_audit}')
records = validate_dataset(dataset_root, config['dataset']['splits'], len(MERGED_CLASS_NAMES), config['dataset']['image_extensions'])
summary = summarize_records(records)
if any(status not in {'valid', 'valid_background'} for status in summary):
    raise RuntimeError(f'Final dataset validation failed: {summary}')
statistics = collect_dataset_statistics(dataset_root, MERGED_CLASS_NAMES, config['dataset']['splits'])
weights = calculate_class_weights(statistics, MERGED_CLASS_NAMES)
person = analyze_person_class(dataset_root, person_id=4, splits=config['dataset']['splits'])
balance = class_balance_summary(statistics)
latest_benchmark = max(reports_dir.glob('memory_benchmark_*.json'), key=lambda path: path.stat().st_mtime)
benchmark = json.loads(latest_benchmark.read_text(encoding='utf-8'))['results']
best_batch = max((record for record in benchmark if record['status'] == 'stable'), key=lambda record: record['batch_size'])
steps_per_epoch = math.ceil(statistics['splits']['train']['images'] / best_batch['batch_size'])
synthetic_train_minutes = statistics['splits']['train']['images'] / best_batch['images_per_second'] / 60
person_review = {
    'person_images': statistics['images_per_class']['Person'],
    'person_annotations': statistics['annotations_per_class']['Person'],
    'person_share_percent': statistics['percentage_per_class']['Person'],
    'object_size_distribution': person,
    'class_balance': balance,
    'class_weights': weights,
    'recommendation': 'Current augmentation is sufficient. Person was increased conservatively to 60% of the smallest non-Person class; adding more synthetic Person samples before a baseline would increase repetition risk without validated benefit.',
    'additional_augmentation_generated': False,
}
hyperparameters = {
    'model': {'value': 'YOLO11s with configurable P3/P4/P5 CBAM', 'reason': 'Retains a compact 9.48M-parameter architecture while targeting multi-scale threat detail; CBAM compute overhead is only 0.016 GFLOPs.'},
    'batch_size': {'value': best_batch['batch_size'], 'reason': 'The measured Apple M4 Pro MPS benchmark was stable through batch 32 at approximately 15.9 GB peak driver memory.'},
    'image_size': {'value': training['training']['image_size'], 'reason': '640 preserves small threat detail while keeping the verified batch-32 memory envelope.'},
    'epochs': {'value': training['training']['epochs'], 'reason': 'A 120-epoch upper bound gives the cosine schedule room to converge; early stopping prevents unnecessary overfitting.'},
    'freeze_epochs': {'value': 0, 'reason': 'Use zero frozen epochs unless a compatible pretrained YOLO11s checkpoint is explicitly configured and verified. Freezing randomly initialized layers would impair learning.'},
    'unfreeze_epochs': {'value': training['training']['epochs'], 'reason': 'All layers should be trainable for a scratch custom model.'},
    'optimizer': {'value': training['training']['optimizer'], 'reason': 'AdamW offers stable adaptive optimization with decoupled weight decay for the attention-augmented architecture.'},
    'learning_rate': {'value': training['training']['learning_rate'], 'reason': '0.003 is a conservative batch-32 AdamW rate for focal loss plus CBAM, limiting unstable large early updates.'},
    'weight_decay': {'value': training['training']['weight_decay'], 'reason': '0.0005 regularizes CBAM and repeated Person augmentation without overly suppressing minority-class learning.'},
    'momentum': {'value': training['training']['momentum'], 'reason': '0.9 is retained for Ultralytics optimizer compatibility; AdamW primarily uses beta parameters internally.'},
    'warmup_epochs': {'value': training['training']['warmup_epochs'], 'reason': 'Three warmup epochs reduce instability from focal modulation and attention layers at initialization.'},
    'scheduler': {'value': training['training']['scheduler'], 'reason': 'Cosine decay supports stable late-stage convergence and reduces overfitting pressure.'},
    'workers': {'value': 6, 'reason': 'Matches the M4 Pro six performance-core-equivalent recommendation while avoiding loader pressure on unified memory.'},
    'cache_mode': {'value': training['training']['cache_mode'], 'reason': 'Disk cache avoids consuming unified memory needed by batch-32 training.'},
    'amp': {'value': training['training']['amp'], 'reason': 'Keep enabled only after a short device-specific AMP smoke confirmation at launch; it is configured for memory efficiency.'},
    'validation_frequency': {'value': training['training']['validation_frequency'], 'reason': 'Validate every epoch to protect the only run and select best.pt using current validation fitness.'},
    'checkpoint_frequency': {'value': training['training']['checkpoint_frequency'], 'reason': 'Save every epoch to recover from interruption on the single permitted run.'},
    'early_stopping_patience': {'value': training['training']['early_stopping_patience'], 'reason': '25 epochs tolerates noisy validation while stopping extended plateau training.'},
    'class_weights': {'value': weights['class_weights'], 'reason': 'Balanced inverse-frequency weights use the post-audit current class distribution.'},
    'focal_loss': {'value': training['loss']['focal'], 'reason': 'Gamma 2.0 and alpha 0.25 remain conservative after Person augmentation; stronger focal settings risk overemphasizing hard minority samples.'},
}
strategy = {
    'checkpointing': 'Save last.pt every epoch and best.pt on validation-fitness improvement. Preserve the entire run directory.',
    'resume': 'Enable resume only from the same approved run directory; Ultralytics checkpoints restore model, optimizer, scheduler, AMP state, epoch, and best metric.',
    'validation': 'Run every epoch and use best.pt for downstream evaluation.',
    'interruption_resilience': 'Every-epoch checkpoints and deterministic configuration make interruption recovery robust once the initialization path is fixed.',
}
overfitting = {
    'dataset_size': 'Large enough to support YOLO11s, but correlated source imagery can still inflate validation metrics.',
    'class_imbalance': f"Post-audit maximum-to-minimum annotation ratio is {balance['max_to_min_ratio']:.2f}; class weights and focal loss should be monitored together.",
    'person_augmentation': 'Conservative augmentation is sufficient; additional repetition is not recommended before a baseline.',
    'cbam': 'Low compute overhead but extra capacity; retain early stopping and per-epoch validation.',
    'focal_loss': 'Use default gamma/alpha; do not increase focal pressure without evidence.',
    'regularization': 'Weight decay, cosine schedule, warmup, and early stopping mitigate overfit risk.',
    'remaining_risk': 'A pretrained initialization path is not configured. This is the critical readiness blocker for a single-run accuracy-focused experiment.',
}
time_estimate = {
    'steps_per_epoch': steps_per_epoch,
    'synthetic_forward_backward_images_per_second': best_batch['images_per_second'],
    'synthetic_training_minutes_per_epoch': synthetic_train_minutes,
    'synthetic_training_hours_for_configured_epochs': synthetic_train_minutes * training['training']['epochs'] / 60,
    'validation_time': 'Not estimated because no validation throughput benchmark was run; measure it before the approved launch rather than guessing.',
    'qualification': 'Synthetic throughput excludes disk I/O, label transforms, validation, checkpointing, and startup overhead.',
}
readiness = {
    'dataset_status': 'pass',
    'duplicate_annotation_status': 'pass',
    'validation_status': 'pass',
    'person_augmentation_status': 'sufficient; no new augmentation generated',
    'model_smoke_status': 'pass from Phase 5',
    'readiness_score': 88,
    'ready_for_final_training': False,
    'blocking_requirement': 'Configure and smoke-test a compatible pretrained YOLO11s initialization path, or explicitly approve scratch training with freeze_epochs=0. Do not begin the only training run until one of these choices is recorded.',
}
stamp = utc_timestamp()
repair_csv = write_duplicate_repair_csv(repair['repairs'], reports_dir / f'duplicate_annotation_repairs_{stamp}.csv')
duplicate_report = write_json(reports_dir / f'duplicate_annotation_audit_{stamp}.json', {'before': before_audit, 'repair': repair, 'after': after_audit, 'repair_csv': str(repair_csv)})
validation_csv = write_validation_csv(records, reports_dir / f'final_readiness_validation_{stamp}.csv')
integrity_report = write_json(reports_dir / f'final_readiness_integrity_{stamp}.json', build_integrity_report(records, config['dataset']['splits']))
statistics_report = write_json(reports_dir / f'final_readiness_statistics_{stamp}.json', statistics)
person_report = write_json(reports_dir / f'final_person_augmentation_review_{stamp}.json', person_review)
hyperparameter_report = write_json(reports_dir / f'final_hyperparameter_recommendations_{stamp}.json', hyperparameters)
strategy_report = write_json(reports_dir / f'final_training_strategy_{stamp}.json', strategy)
overfitting_report = write_json(reports_dir / f'final_overfitting_analysis_{stamp}.json', overfitting)
time_report = write_json(reports_dir / f'final_training_time_estimation_{stamp}.json', time_estimate)
readiness_report = write_json(reports_dir / f'phase_6_final_readiness_{stamp}.json', {'duplicate_audit': str(duplicate_report), 'validation_csv': str(validation_csv), 'integrity_report': str(integrity_report), 'statistics': str(statistics_report), 'person_augmentation_review': str(person_report), 'hyperparameter_recommendations': str(hyperparameter_report), 'training_strategy': str(strategy_report), 'overfitting_analysis': str(overfitting_report), 'training_time_estimation': str(time_report), 'readiness': readiness, 'training_started': False})
logger.info('Duplicate audit completed: %s', after_audit)
logger.info('Final validation completed: %s', summary)
logger.info('Training readiness: %s', readiness)
print({'duplicate_repair': repair, 'validation': summary, 'readiness': readiness, 'report': str(readiness_report)})

{'duplicate_repair': {'files_scanned': 131467, 'files_repaired': 137, 'duplicates_removed': 139, 'repairs': [{'split': 'train', 'label': '/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels/00000389_jpg.rf.bd357afc09858f6968f9780e0f780eae.txt', 'removed_line_numbers': [2], 'duplicates_removed': 1}, {'split': 'train', 'label': '/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels/01736917d1d06920_jpg.rf.3aa46698efa6b77a52f71019132b51e3.txt', 'removed_line_numbers': [2], 'duplicates_removed': 1}, {'split': 'train', 'label': '/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels/02792cd530f2d505_jpg.rf.a42d460416bad934d2eb2211fae267bf.txt', 'removed_line_numbers': [4], 'duplicates_removed': 1}, {'split': 'train', 'label': '/Users/mohamedtarek/weapon130/processed/final_dataset_v1/train/labels/0a8ab3f65ad3b6a6_jpg.rf.30dd35fe4bcd4a14e2994059747514ef.txt', 'removed_line_numbers': [2], 'duplicates_removed': 1}, {'split': 'train', 'label': '/U